In [1]:
import kfp.dsl as dsl
from kfp.v2 import compiler
from kfp.v2.dsl import component,Model,Output
from typing import NamedTuple
from kfp.v2.dsl import Input, Model, Output, component

In [ ]:
PROJECT_ID = "XXXXX"
BUCKET_NAME = 'XXXXX'
PIPELINE_NAME = 'pipeline-ml_model_auto_build'
PIPELINE_ROOT = 'gs://XXXXX/pipeline_root/XXXXX/'

### データの取得

In [30]:
@component(
    base_image='gcr.io/deeplearning-platform-release/sklearn-cpu:latest'
)
def get_data_op(transformed_data_path: str)-> NamedTuple("outputs", [("transformed_data_path", str)]):
    from sklearn.datasets import fetch_california_housing
    from sklearn.model_selection import train_test_split
    import os
    import pandas as pd
    california = fetch_california_housing()
    x = california.data 
    y = california.target 
    x_train, x_valid, y_train, y_valid = train_test_split(x, y, test_size=0.2, random_state=42)
    # output
    pd.DataFrame(y_train).to_csv(transformed_data_path + "/y_train.csv")
    pd.DataFrame(x_train).to_csv(transformed_data_path + "/x_train.csv")
    pd.DataFrame(y_valid).to_csv(transformed_data_path + "/y_valid.csv")
    pd.DataFrame(x_valid).to_csv(transformed_data_path + "/x_valid.csv")
    return (transformed_data_path,)

### モデルの学習とdump

In [31]:
@component(
    base_image='gcr.io/deeplearning-platform-release/sklearn-cpu:latest'
)
def train_model_op(transformed_data_path:str,trained_model_path:str,output_model: Output[Model]):
    from sklearn import linear_model
    import pandas as pd
    import os
    import pickle
    # input
    y_train = pd.read_csv(transformed_data_path + "/y_train.csv", index_col=0)
    x_train = pd.read_csv(transformed_data_path + "/x_train.csv", index_col=0)    
    model = linear_model.LinearRegression()
    model.fit(x_train, y_train)
    # output
    os.makedirs(output_model.path, exist_ok=True)
    with open(os.path.join(output_model.path, 'model.pkl'), "wb") as f:
        pickle.dump(model, f)

### モデルの精度検証

In [32]:
@component(
    base_image='gcr.io/deeplearning-platform-release/sklearn-cpu:latest'
)
def eval_model_op(
    #trained_model_path:str
    input_model:Input[Model]
    ,transformed_data_path:str
    , bucket_name:str
) -> None:
    import pandas as pd
    import pickle
    import os 
    # input
    y_valid = pd.read_csv(transformed_data_path + "/y_valid.csv", index_col=0)
    x_valid = pd.read_csv(transformed_data_path + "/x_valid.csv", index_col=0)
    y_train = pd.read_csv(transformed_data_path + "/y_train.csv", index_col=0)
    x_train = pd.read_csv(transformed_data_path + "/x_train.csv", index_col=0)
    model = pickle.load(open(input_model.path + "/model.pkl", "rb"))
    df = pd.DataFrame(
        {
            'train_score': [model.score(x_train, y_train)],
            'test_score': [model.score(x_valid, y_valid)]
        }
    )
    df.to_csv('gs://{}/XXXXXX/result.csv'.format(bucket_name), index=False)

### テストスコアをBQに格納

In [33]:
@component(
    base_image='gcr.io/deeplearning-platform-release/sklearn-cpu:latest'
)
def export_result_to_bq_op(project_id:str="", bucket_name:str="")->None:
    from google.cloud import bigquery
    client = bigquery.Client(project=project_id)
    table_id='{}.vertexai.boston_eval'.format(project_id)
    job_config = bigquery.LoadJobConfig(
        schema=[
            bigquery.SchemaField("train_score", "FLOAT"),
            bigquery.SchemaField("test_score", "FLOAT"),
        ],
        skip_leading_rows=1,
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        source_format=bigquery.SourceFormat.CSV,
    )
    uri = "gs://{}/XXXXX/result.csv".format(bucket_name)
    load_job = client.load_table_from_uri(
        uri, table_id, job_config=job_config
    )
    load_job.result()

### パイプラインのビルド

In [34]:
# パイプライン構築
@dsl.pipeline(
    name=PIPELINE_NAME,
    description='VertexAIPipeline',
    pipeline_root=PIPELINE_ROOT
)
def pipeline(PROJECT_ID:str,BUCKET_NAME:str,PIPELINE_ROOT:str):
    get_data = get_data_op(transformed_data_path=PIPELINE_ROOT)
    train_model = train_model_op(get_data.outputs["transformed_data_path"],trained_model_path=PIPELINE_ROOT)
    train_model.after(get_data)
    eval_model = eval_model_op(
        input_model=train_model.outputs["output_model"]
        ,transformed_data_path=get_data.outputs["transformed_data_path"]
        ,bucket_name=BUCKET_NAME
    )
    # 依存関係の定義
    export_result_to_bq_op(project_id=PROJECT_ID, bucket_name=BUCKET_NAME).after(eval_model)

In [ ]:
# パイプラインのコンパイル
compiler.Compiler().compile(pipeline_func=pipeline, package_path='./VertexAIPipeline.json')

In [36]:
from datetime import datetime
TIMESTAMP = datetime.now().strftime("%Y%m%d%H%M%S")

from google.cloud import aiplatform
pipeline_job = aiplatform.PipelineJob(
    display_name="VertexAIPipeline",
    template_path="VertexAIPipeline.json",
    job_id="custom-boston-pipeline-{}".format(TIMESTAMP),
    parameter_values={
        "PROJECT_ID":PROJECT_ID,
        "BUCKET_NAME":BUCKET_NAME,
        "PIPELINE_ROOT":PIPELINE_ROOT
    },
    enable_caching=False,
)

In [ ]:
# パイプラインの実行
pipeline_job.submit()